# Hampton Roads Flood Risk And Climate Impact Report

This notebook is the report runner for the final 7-city regional analysis. It refreshes the database-derived reports, loads the generated CSVs, and displays the tables most useful for the final writeup or presentation.

The core spatial model answers: what roads, buildings, and property-value proxies are exposed if water reaches each modeled elevation? The climate framing added here answers a separate question: how should each water-level scenario be interpreted as a current event, future sea-level-rise planning case, or stress-test possibility?

Frozen regional scope: Norfolk, Virginia Beach, Chesapeake, Hampton, Newport News, Portsmouth, and Suffolk.


## How To Use

Run the notebook top to bottom from the repository root. The heavy spatial workflow should already have been run. This notebook refreshes the final report CSVs from the database and displays them.

If another process is still building GIS data, leave `RUN_FULL_0_TO_6FT_WORKFLOW = False` and `REFRESH_GIS_EXPORTS = False`. The climate-possibility tables use whichever report rows currently exist and will automatically include additional SLR scenarios after the GIS workflow finishes and the report CSVs are refreshed.

If you need to regenerate the city GeoPackages too, set `REFRESH_GIS_EXPORTS = True` in the setup cell.


## 1. Setup


In [1]:
from __future__ import annotations

import subprocess
import sys
from pathlib import Path

import pandas as pd
from sqlalchemy import text

from flood_analysis.db import get_engine

PROJECT_ROOT = Path.cwd()
REPORT_DIR = PROJECT_ROOT / "data" / "processed" / "gis"
ACS_YEAR = 2023
TOP_ROAD_LIMIT = 10
REFRESH_ACS_VALUES = False
REFRESH_GIS_EXPORTS = False
RUN_FULL_0_TO_6FT_WORKFLOW = False
FUTURE_SLR_DECADES = {
    2030: 0.5,
    2040: 1.0,
    2050: 1.5,
    2060: 2.0,
    2070: 2.5,
    2080: 3.0,
    2090: 3.75,
    2100: 4.5,
}
FULL_SLR_VALUES = ["0", "0.5", "1", "1.5", "2", "2.5", "3", "3.75", "4", "4.5", "5", "6"]
NORFOLK_FULL_SCENARIO_IDS = [
    f"norfolk_pilot_8638610_20260627_plus_{str(float(value)).replace('.', 'p')}ft"
    for value in FULL_SLR_VALUES
]
NORFOLK_SCENARIO_ARGS = [item for scenario_id in NORFOLK_FULL_SCENARIO_IDS for item in ("--scenario-id", scenario_id)]

CITY_EXPORTS = {
    "norfolk_va": "norfolk_flood_exposure_1m.gpkg",
    "virginia_beach_va": "virginia_beach_flood_exposure_1m.gpkg",
    "chesapeake_va": "chesapeake_flood_exposure_1m.gpkg",
    "hampton_va": "hampton_flood_exposure_1m.gpkg",
    "newport_news_va": "newport_news_flood_exposure_1m.gpkg",
    "portsmouth_va": "portsmouth_flood_exposure_1m.gpkg",
    "suffolk_va": "suffolk_flood_exposure_1m.gpkg",
}

REPORT_DIR.mkdir(parents=True, exist_ok=True)
print(f"project_root={PROJECT_ROOT}")
print(f"report_dir={REPORT_DIR}")
print(f"full_slr_values_ft={' '.join(FULL_SLR_VALUES)}")


project_root=/home/rthomson/odu/cs620/project
report_dir=/home/rthomson/odu/cs620/project/data/processed/gis
full_slr_values_ft=0 0.5 1 1.5 2 2.5 3 3.75 4 4.5 5 6


In [2]:
def run_command(args: list[str]) -> None:
    command = [str(part) for part in args]
    print("$", " ".join(command))
    completed = subprocess.run(command, cwd=PROJECT_ROOT, text=True, capture_output=True)
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr, file=sys.stderr)
    completed.check_returncode()

def money(value: float) -> str:
    return f"${value:,.0f}"

def load_report(name: str) -> pd.DataFrame:
    path = REPORT_DIR / name
    if not path.exists():
        raise FileNotFoundError(f"Missing report: {path}")
    return pd.read_csv(path)

def format_money_columns(frame: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    formatted = frame.copy()
    for column in columns:
        if column in formatted.columns:
            formatted[column] = formatted[column].map(money)
    return formatted

def climate_possibility_band(sea_level_rise_ft: float) -> str:
    value = round(float(sea_level_rise_ft), 2)
    if value == 0:
        return "Observed event benchmark"
    if value <= 1:
        return "Near-term climate possibility"
    if value <= 2:
        return "Mid-century climate possibility"
    if value <= 3:
        return "Late-century high-impact possibility"
    if value <= 4.5:
        return "Long-range high-impact possibility"
    return "Upper stress-test possibility"

def add_climate_possibility(frame: pd.DataFrame) -> pd.DataFrame:
    labeled = frame.copy()
    labeled["climate_possibility"] = labeled["sea_level_rise_ft"].map(climate_possibility_band)
    labeled["probability_note"] = labeled["scenario_type"].map({
        "current_event": "Observed NOAA event baseline, not a recurrence estimate",
        "future_decade": "Scenario-based SLR planning case, not an annual probability",
        "stress_view": "Sensitivity stress view, not a dated forecast",
    }).fillna("Scenario interpretation only")
    return labeled

## 2. Optional Full 0-6 ft Workflow

The established model already supports arbitrary sea-level-rise increments. Set `RUN_FULL_0_TO_6FT_WORKFLOW = True` in the setup cell to rebuild flood views from current conditions through `+6 ft` SLR, including the decade planning curve below. This is heavy because it regenerates rasters, connected extents, road exposure, building exposure, damage, property-value exposure, and report CSVs.

Planning curve: 2030 `+0.5 ft`, 2040 `+1.0 ft`, 2050 `+1.5 ft`, 2060 `+2.0 ft`, 2070 `+2.5 ft`, 2080 `+3.0 ft`, 2090 `+3.75 ft`, 2100 `+4.5 ft`. The `+4`, `+5`, and `+6 ft` outputs are additional stress-view flood layers that account for flooding on top of sea-level rise using the same event baseline.


In [3]:
if RUN_FULL_0_TO_6FT_WORKFLOW:
    run_command([
        sys.executable,
        "scripts/create_peak_scenarios.py",
        "--event-start",
        "2026-06-27T00:00:00Z",
        "--event-end",
        "2026-06-28T23:59:59Z",
        "--study-area",
        "Norfolk pilot",
        "--sea-level-rise-ft",
        *FULL_SLR_VALUES,
        "--method",
        "1-meter Norfolk screening; connected flood extents; 0-6 ft SLR views",
        "--notes",
        "Flood views through +6 ft, including decade SLR planning curve through 2100.",
    ])
    run_command([sys.executable, "scripts/convert_scenarios_datum.py", "--station", "8638610", "--source-datum", "MLLW", "--target-datum", "NAVD88"])
    run_command([
        sys.executable,
        "scripts/create_flood_depth_rasters.py",
        "--study-area-id",
        "norfolk_va",
        "--dem",
        "data/processed/dem/norfolk_va_usgs_1m_hamptonroads_b23_navd88_m.tif",
        "--dem-units",
        "meters",
        "--output-dir",
        "data/processed/flood_depths_1m",
        *NORFOLK_SCENARIO_ARGS,
    ])
    run_command([sys.executable, "scripts/polygonize_flood_extents.py", "--min-depth-ft", "0", *NORFOLK_SCENARIO_ARGS])
    run_command([sys.executable, "scripts/create_connected_flood_depth_rasters.py", "--output-dir", "data/processed/flood_depths_connected_1m", "--min-depth-ft", "0", *NORFOLK_SCENARIO_ARGS])
    run_command([sys.executable, "scripts/polygonize_connected_flood_extents.py", "--min-depth-ft", "0", *NORFOLK_SCENARIO_ARGS])
    run_command([sys.executable, "scripts/calculate_road_exposure.py", "--study-area-id", "norfolk_va", "--connected"])
    run_command([sys.executable, "scripts/calculate_building_exposure.py", "--study-area-id", "norfolk_va"])
    run_command([sys.executable, "scripts/calculate_building_damage.py", "--study-area-id", "norfolk_va", "--replacement-cost-per-sqft", "175"])
    for study_area_id in [key for key in CITY_EXPORTS if key != "norfolk_va"]:
        run_command([sys.executable, "scripts/run_regional_coarse_workflow.py", "--study-area-id", study_area_id, "--sea-level-rise-ft", *FULL_SLR_VALUES])
else:
    print("Skipping full 0-6 ft rebuild. Set RUN_FULL_0_TO_6FT_WORKFLOW = True to generate expanded flood views.")

Skipping full 0-6 ft rebuild. Set RUN_FULL_0_TO_6FT_WORKFLOW = True to generate expanded flood views.


## 3. Refresh Report CSVs

These commands update ACS property-value exposure and regenerate all presentation-ready CSV outputs.


In [4]:
property_value_command = [sys.executable, "scripts/calculate_property_value_exposure.py", "--year", str(ACS_YEAR)]
if not REFRESH_ACS_VALUES:
    property_value_command.append("--skip-fetch")
run_command(property_value_command)
run_command([
    sys.executable,
    "scripts/export_presentation_outputs.py",
    "--output-dir",
    str(REPORT_DIR),
    "--top-road-limit",
    str(TOP_ROAD_LIMIT),
])

$ /home/rthomson/odu/cs620/project/.venv/bin/python scripts/calculate_property_value_exposure.py --year 2023 --skip-fetch


scenario_id=suffolk_pilot_8638610_20260627_plus_3p75ft study_area_id=suffolk_va estimated_exposed_property_value=$92,989
scenario_id=suffolk_pilot_8638610_20260627_plus_4p0ft study_area_id=suffolk_va estimated_exposed_property_value=$125,377
scenario_id=suffolk_pilot_8638610_20260627_plus_4p5ft study_area_id=suffolk_va estimated_exposed_property_value=$360,048
scenario_id=suffolk_pilot_8638610_20260627_plus_5p0ft study_area_id=suffolk_va estimated_exposed_property_value=$417,386
scenario_id=suffolk_pilot_8638610_20260627_plus_6p0ft study_area_id=suffolk_va estimated_exposed_property_value=$2,820,979
scenario_id=hampton_pilot_8638610_20260627_plus_0p0ft study_area_id=hampton_va estimated_exposed_property_value=$11,003,907
scenario_id=hampton_pilot_8638610_20260627_plus_0p5ft study_area_id=hampton_va estimated_exposed_property_value=$12,003,252
scenario_id=portsmouth_pilot_8638610_20260627_plus_0p0ft study_area_id=portsmouth_va estimated_exposed_property_value=$14,995,937
scenario_id=ham

exported=/home/rthomson/odu/cs620/project/data/processed/gis/regional_flood_comparison.csv rows=84
exported=/home/rthomson/odu/cs620/project/data/processed/gis/regional_top_impacted_roads.csv rows=749
exported=/home/rthomson/odu/cs620/project/data/processed/gis/regional_chart_damage_by_city_scenario.csv
exported=/home/rthomson/odu/cs620/project/data/processed/gis/regional_chart_flooded_buildings_by_city_scenario.csv
exported=/home/rthomson/odu/cs620/project/data/processed/gis/regional_chart_flooded_road_miles_by_city_scenario.csv
exported=/home/rthomson/odu/cs620/project/data/processed/gis/regional_chart_exposed_property_value_by_city_scenario.csv
exported=/home/rthomson/odu/cs620/project/data/processed/gis/regional_chart_all_slr_summary.csv
exported=/home/rthomson/odu/cs620/project/data/processed/gis/regional_chart_decade_summary.csv
exported=/home/rthomson/odu/cs620/project/data/processed/gis/regional_chart_2100_summary.csv
exported=/home/rthomson/odu/cs620/project/data/processed/gis

In [5]:
if REFRESH_GIS_EXPORTS:
    for study_area_id, filename in CITY_EXPORTS.items():
        run_command([
            sys.executable,
            "scripts/export_gis_layers.py",
            "--study-area-id",
            study_area_id,
            "--output",
            str(REPORT_DIR / filename),
        ])
else:
    print("Skipping GeoPackage refresh. Set REFRESH_GIS_EXPORTS = True to regenerate GIS exports.")

Skipping GeoPackage refresh. Set REFRESH_GIS_EXPORTS = True to regenerate GIS exports.


## 4. Load Reports


In [6]:
regional = load_report("regional_flood_comparison.csv")
plus_3ft = load_report("regional_chart_plus_3ft_summary.csv")
year_2100 = load_report("regional_chart_2100_summary.csv")
plus_6ft = load_report("regional_chart_plus_6ft_summary.csv")
scenario_lookup = load_report("regional_scenario_lookup.csv")
metric_summary = load_report("regional_metric_summary.csv")
plus_3ft_metrics = load_report("regional_plus_3ft_metric_summary.csv")
top_roads = load_report("regional_top_impacted_roads.csv")
damage_chart = load_report("regional_chart_damage_by_city_scenario.csv")
buildings_chart = load_report("regional_chart_flooded_buildings_by_city_scenario.csv")
roads_chart = load_report("regional_chart_flooded_road_miles_by_city_scenario.csv")
property_chart = load_report("regional_chart_exposed_property_value_by_city_scenario.csv")

print(f"regional_rows={len(regional)}")
print(f"top_impacted_road_rows={len(top_roads)}")

regional_rows=84
top_impacted_road_rows=749


## Executive Summary

This report separates direct flood-impact metrics from climate/sea-level-rise planning metrics. Flood-impact results describe what is exposed if a modeled water surface occurs. Climate/SLR results describe how impacts grow across planning increments and stress views, not annual probability or return period.


In [7]:
headline_scenarios = [3.0, 4.5, 6.0]
headline = regional[regional["sea_level_rise_ft"].isin(headline_scenarios)].groupby("sea_level_rise_ft", as_index=False).agg(
    flooded_buildings=("flooded_building_count", "sum"),
    flooded_road_miles=("flooded_road_miles", "sum"),
    depth_based_damage_cost=("estimated_damage_cost", "sum"),
    exposed_property_value_proxy=("estimated_exposed_property_value", "sum"),
)
headline["scenario_label"] = headline["sea_level_rise_ft"].map(
    scenario_lookup.set_index("sea_level_rise_ft")["scenario_label"].to_dict()
)
headline = headline[[
    "scenario_label",
    "sea_level_rise_ft",
    "flooded_buildings",
    "flooded_road_miles",
    "depth_based_damage_cost",
    "exposed_property_value_proxy",
]]
headline_display = headline.copy()
headline_display["flooded_road_miles"] = headline_display["flooded_road_miles"].round(2)
for column in ["depth_based_damage_cost", "exposed_property_value_proxy"]:
    headline_display[column] = headline_display[column].map(money)
headline_display


,scenario_label,sea_level_rise_ft,flooded_buildings,flooded_road_miles,depth_based_damage_cost,exposed_property_value_proxy
0,2080 (+3 ft SLR),3.0,9148,379.33,"$510,049,254","$2,966,097,995"
1,2100 (+4.5 ft SLR),4.5,24256,721.89,"$1,362,343,370","$7,882,478,119"
2,+6 ft SLR stress view,6.0,57606,1367.24,"$3,733,834,890","$19,028,116,065"


In [8]:
top_damage_city = regional[regional["sea_level_rise_ft"].isin(headline_scenarios)].sort_values(
    ["sea_level_rise_ft", "estimated_damage_cost"],
    ascending=[True, False],
).groupby("sea_level_rise_ft", as_index=False).head(1)
top_damage_display = top_damage_city[[
    "scenario_label",
    "study_area_name",
    "flooded_building_count",
    "flooded_road_miles",
    "estimated_damage_cost",
    "estimated_exposed_property_value",
]].copy()
top_damage_display["flooded_road_miles"] = top_damage_display["flooded_road_miles"].round(2)
for column in ["estimated_damage_cost", "estimated_exposed_property_value"]:
    top_damage_display[column] = top_damage_display[column].map(money)
top_damage_display


,scenario_label,study_area_name,flooded_building_count,flooded_road_miles,estimated_damage_cost,estimated_exposed_property_value
42,2080 (+3 ft SLR),"Norfolk, VA",1773,53.54,"$223,492,812","$485,979,044"
81,2100 (+4.5 ft SLR),"Virginia Beach, VA",10221,302.44,"$467,888,651","$3,860,876,685"
83,+6 ft SLR stress view,"Virginia Beach, VA",23797,509.93,"$1,237,911,533","$9,115,664,110"


## How To Read This Report

Flood impact metrics summarize buildings, roads, damage, property-value proxy exposure, and recovery categories for each modeled water surface. Climate/SLR planning metrics use the same outputs but organize them by decade, SLR increment, stress view, and per-foot sensitivity. Priority corridor metrics are screening-level flooded road-centerline miles for recognizable routes; they should not be interpreted as engineering road-closure predictions.


## 5. Data Coverage Checks

Before interpreting results, verify that the report CSVs contain the frozen 7-city scope and the full `0-6 ft` scenario set. The property-value check flags rows where flooded buildings exist but ACS proxy exposure is missing or zero.



In [9]:
EXPECTED_CITY_COUNT = len(CITY_EXPORTS)
EXPECTED_SLR_VALUES = [float(value) for value in FULL_SLR_VALUES]

coverage_by_slr = regional.groupby("sea_level_rise_ft", as_index=False).agg(
    city_count=("study_area_id", "nunique"),
    rows=("study_area_id", "count"),
    flooded_buildings=("flooded_building_count", "sum"),
    flooded_road_miles=("flooded_road_miles", "sum"),
    estimated_damage_cost=("estimated_damage_cost", "sum"),
    estimated_exposed_property_value=("estimated_exposed_property_value", "sum"),
)
coverage_by_slr["expected_city_count"] = EXPECTED_CITY_COUNT
coverage_by_slr["complete_city_coverage"] = coverage_by_slr["city_count"] == EXPECTED_CITY_COUNT

available_slr = sorted(regional["sea_level_rise_ft"].dropna().unique().tolist())
missing_slr = sorted(set(EXPECTED_SLR_VALUES) - set(available_slr))
property_value_gaps = regional[
    (regional["flooded_building_count"] > 0)
    & (regional["estimated_exposed_property_value"].fillna(0) <= 0)
].copy()

print(f"city_count={regional['study_area_id'].nunique()} expected={EXPECTED_CITY_COUNT}")
print(f"scenario_count={len(available_slr)} expected={len(EXPECTED_SLR_VALUES)}")
print(f"missing_slr_values={missing_slr}")
print(f"property_value_gap_rows={len(property_value_gaps)}")

display_columns = [
    "sea_level_rise_ft",
    "city_count",
    "rows",
    "complete_city_coverage",
    "flooded_buildings",
    "flooded_road_miles",
    "estimated_damage_cost",
    "estimated_exposed_property_value",
]
coverage_display = coverage_by_slr[display_columns].copy()
coverage_display["flooded_road_miles"] = coverage_display["flooded_road_miles"].round(2)
for column in ["estimated_damage_cost", "estimated_exposed_property_value"]:
    coverage_display[column] = coverage_display[column].map(money)
coverage_display


city_count=7 expected=7
scenario_count=12 expected=12
missing_slr_values=[]
property_value_gap_rows=7


,sea_level_rise_ft,city_count,rows,complete_city_coverage,flooded_buildings,flooded_road_miles,estimated_damage_cost,estimated_exposed_property_value
0,0.00,7,7,True,1025,129.86,"$163,967,589","$320,444,366"
1,0.50,7,7,True,1422,153.02,"$181,009,191","$466,948,803"
2,1.00,7,7,True,2065,183.76,"$203,879,561","$712,993,009"
3,1.50,7,7,True,2911,217.40,"$227,241,043","$998,860,733"
4,2.00,7,7,True,4173,255.45,"$263,004,538","$1,411,100,175"
5,2.50,7,7,True,6242,304.34,"$385,481,100","$2,062,104,014"
6,3.00,7,7,True,9148,379.33,"$510,049,254","$2,966,097,995"
7,3.75,7,7,True,15324,512.76,"$826,350,687","$4,957,553,138"
8,4.00,7,7,True,17922,573.08,"$975,964,316","$5,807,343,909"
9,4.50,7,7,True,24256,721.89,"$1,362,343,370","$7,882,478,119"


## Metric Family Split

The report now separates deterministic flood-impact metrics from climate/sea-level-rise planning metrics. Flood-impact metrics describe what is exposed at a modeled water surface. Climate-planning metrics describe how those flood impacts change across future SLR planning increments and stress views.


In [10]:
metric_family_rows = [
    {
        "metric_family": "Flood impact",
        "metric": "Flooded buildings",
        "source_fields": "flooded_building_count, flooded_building_fraction",
        "interpretation": "Buildings with positive connected flood exposure in a modeled flood layer.",
    },
    {
        "metric_family": "Flood impact",
        "metric": "Flooded roads",
        "source_fields": "flooded_road_count, flooded_road_miles",
        "interpretation": "Road segments and mileage intersecting connected flood extents.",
    },
    {
        "metric_family": "Flood impact",
        "metric": "Depth-based structure damage",
        "source_fields": "estimated_damage_cost, average_damage_cost, max_estimated_recovery_days",
        "interpretation": "Screening structural damage from flood depth, footprint area, and replacement-cost assumptions.",
    },
    {
        "metric_family": "Flood impact",
        "metric": "ACS property-value exposure proxy",
        "source_fields": "median_home_value, estimated_exposed_property_value",
        "interpretation": "City-level median owner-occupied home value multiplied by flooded building area fraction.",
    },
    {
        "metric_family": "Climate / SLR planning",
        "metric": "Planning decade",
        "source_fields": "planning_year, scenario_type, scenario_label",
        "interpretation": "Scenario framing for SLR increments from 2030 through 2100; not annual probability.",
    },
    {
        "metric_family": "Climate / SLR planning",
        "metric": "Per-foot SLR sensitivity",
        "source_fields": "metric deltas divided by SLR interval",
        "interpretation": "How quickly impacts increase between modeled SLR increments.",
    },
    {
        "metric_family": "Climate / SLR planning",
        "metric": "Stress-view escalation",
        "source_fields": "+3 ft, +4.5 ft, +6 ft comparisons",
        "interpretation": "Scenario stress testing above the presentation benchmark and 2100 planning case.",
    },
]
metric_family_inventory = pd.DataFrame(metric_family_rows)
metric_family_inventory


,metric_family,metric,source_fields,interpretation
0,Flood impact,Flooded buildings,"flooded_building_count, flooded_building_fraction",Buildings with positive connected flood exposu...
1,Flood impact,Flooded roads,"flooded_road_count, flooded_road_miles",Road segments and mileage intersecting connect...
2,Flood impact,Depth-based structure damage,"estimated_damage_cost, average_damage_cost, ma...","Screening structural damage from flood depth, ..."
3,Flood impact,ACS property-value exposure proxy,"median_home_value, estimated_exposed_property_...",City-level median owner-occupied home value mu...
4,Climate / SLR planning,Planning decade,"planning_year, scenario_type, scenario_label",Scenario framing for SLR increments from 2030 ...
5,Climate / SLR planning,Per-foot SLR sensitivity,metric deltas divided by SLR interval,How quickly impacts increase between modeled S...
6,Climate / SLR planning,Stress-view escalation,"+3 ft, +4.5 ft, +6 ft comparisons",Scenario stress testing above the presentation...


## Flood Impact Metrics

These tables treat each water level as a flood layer and summarize direct exposure and damage, without assigning climate probability or return period.


In [11]:
flood_impact_summary = regional.groupby("sea_level_rise_ft", as_index=False).agg(
    city_count=("study_area_id", "nunique"),
    flooded_buildings=("flooded_building_count", "sum"),
    flooded_building_share_mean=("flooded_building_fraction", "mean"),
    flooded_road_count=("flooded_road_count", "sum"),
    flooded_road_miles=("flooded_road_miles", "sum"),
    damaged_buildings=("damaged_building_count", "sum"),
    depth_based_damage_cost=("estimated_damage_cost", "sum"),
    exposed_property_value_proxy=("estimated_exposed_property_value", "sum"),
    max_recovery_days=("max_estimated_recovery_days", "max"),
)
flood_impact_summary["scenario_label"] = flood_impact_summary["sea_level_rise_ft"].map(
    scenario_lookup.set_index("sea_level_rise_ft")["scenario_label"].to_dict()
)
flood_impact_display = flood_impact_summary[[
    "scenario_label",
    "sea_level_rise_ft",
    "city_count",
    "flooded_buildings",
    "flooded_building_share_mean",
    "flooded_road_count",
    "flooded_road_miles",
    "damaged_buildings",
    "depth_based_damage_cost",
    "exposed_property_value_proxy",
    "max_recovery_days",
]].copy()
flood_impact_display["flooded_building_share_mean"] = (flood_impact_display["flooded_building_share_mean"] * 100).round(2)
flood_impact_display["flooded_road_miles"] = flood_impact_display["flooded_road_miles"].round(2)
for column in ["depth_based_damage_cost", "exposed_property_value_proxy"]:
    flood_impact_display[column] = flood_impact_display[column].map(money)
flood_impact_display


,scenario_label,sea_level_rise_ft,city_count,flooded_buildings,flooded_building_share_mean,flooded_road_count,flooded_road_miles,damaged_buildings,depth_based_damage_cost,exposed_property_value_proxy,max_recovery_days
0,Current event,0.00,7,1025,0.13,1013,129.86,1024,"$163,967,589","$320,444,366",180
1,2030 (+0.5 ft SLR),0.50,7,1422,0.16,1227,153.02,1421,"$181,009,191","$466,948,803",180
2,2040 (+1 ft SLR),1.00,7,2065,0.22,1457,183.76,2063,"$203,879,561","$712,993,009",180
3,2050 (+1.5 ft SLR),1.50,7,2911,0.31,1738,217.40,2909,"$227,241,043","$998,860,733",180
4,2060 (+2 ft SLR),2.00,7,4173,0.46,2053,255.45,4169,"$263,004,538","$1,411,100,175",180
5,2070 (+2.5 ft SLR),2.50,7,6242,0.75,2453,304.34,6236,"$385,481,100","$2,062,104,014",180
6,2080 (+3 ft SLR),3.00,7,9148,1.17,2959,379.33,9140,"$510,049,254","$2,966,097,995",180
7,2090 (+3.75 ft SLR),3.75,7,15324,2.04,3841,512.76,15324,"$826,350,687","$4,957,553,138",180
8,+4 ft SLR stress view,4.00,7,17922,2.40,4248,573.08,17922,"$975,964,316","$5,807,343,909",180
9,2100 (+4.5 ft SLR),4.50,7,24256,3.30,5285,721.89,24256,"$1,362,343,370","$7,882,478,119",180


In [12]:
flood_city_rankings = regional[regional["sea_level_rise_ft"].isin([3.0, 4.5, 6.0])].copy()
flood_city_rankings = flood_city_rankings.sort_values(
    ["sea_level_rise_ft", "estimated_damage_cost"],
    ascending=[True, False],
)
flood_city_rankings_display = flood_city_rankings[[
    "scenario_label",
    "study_area_name",
    "flooded_building_count",
    "flooded_road_miles",
    "estimated_damage_cost",
    "estimated_exposed_property_value",
    "max_estimated_recovery_days",
]].copy()
flood_city_rankings_display["flooded_road_miles"] = flood_city_rankings_display["flooded_road_miles"].round(2)
for column in ["estimated_damage_cost", "estimated_exposed_property_value"]:
    flood_city_rankings_display[column] = flood_city_rankings_display[column].map(money)
flood_city_rankings_display.groupby("scenario_label", as_index=False).head(3)


,scenario_label,study_area_name,flooded_building_count,flooded_road_miles,estimated_damage_cost,estimated_exposed_property_value,max_estimated_recovery_days
42,2080 (+3 ft SLR),"Norfolk, VA",1773,53.54,"$223,492,812","$485,979,044",180
78,2080 (+3 ft SLR),"Virginia Beach, VA",4464,176.38,"$153,778,430","$1,676,290,498",180
18,2080 (+3 ft SLR),"Hampton, VA",1350,66.67,"$48,409,085","$323,146,760",180
81,2100 (+4.5 ft SLR),"Virginia Beach, VA",10221,302.44,"$467,888,651","$3,860,876,685",180
45,2100 (+4.5 ft SLR),"Norfolk, VA",4458,97.05,"$410,965,031","$1,265,139,979",180
21,2100 (+4.5 ft SLR),"Hampton, VA",4235,151.28,"$217,338,862","$1,033,022,770",180
83,+6 ft SLR stress view,"Virginia Beach, VA",23797,509.93,"$1,237,911,533","$9,115,664,110",180
47,+6 ft SLR stress view,"Norfolk, VA",8703,217.76,"$920,562,748","$2,509,389,882",180
23,+6 ft SLR stress view,"Hampton, VA",11045,271.76,"$699,438,358","$2,838,759,785",180


## Climate / SLR Planning Metrics

These metrics interpret the same flood outputs across the SLR planning curve. They do not estimate annual probability; they show impact growth between planning increments.


In [13]:
decade_planning = regional[regional["scenario_type"] == "future_decade"].copy()
decade_planning_summary = decade_planning.groupby(["planning_year", "sea_level_rise_ft", "scenario_label"], as_index=False).agg(
    flooded_buildings=("flooded_building_count", "sum"),
    flooded_road_miles=("flooded_road_miles", "sum"),
    depth_based_damage_cost=("estimated_damage_cost", "sum"),
    exposed_property_value_proxy=("estimated_exposed_property_value", "sum"),
)
decade_planning_summary = decade_planning_summary.sort_values("planning_year")
for metric in ["flooded_buildings", "flooded_road_miles", "depth_based_damage_cost", "exposed_property_value_proxy"]:
    decade_planning_summary[f"additional_{metric}_since_prior_decade"] = decade_planning_summary[metric].diff()
climate_decade_display = decade_planning_summary.copy()
for column in ["flooded_road_miles", "additional_flooded_road_miles_since_prior_decade"]:
    climate_decade_display[column] = climate_decade_display[column].round(2)
for column in [
    "depth_based_damage_cost",
    "exposed_property_value_proxy",
    "additional_depth_based_damage_cost_since_prior_decade",
    "additional_exposed_property_value_proxy_since_prior_decade",
]:
    climate_decade_display[column] = climate_decade_display[column].fillna(0).map(money)
climate_decade_display


,planning_year,sea_level_rise_ft,scenario_label,flooded_buildings,flooded_road_miles,depth_based_damage_cost,exposed_property_value_proxy,additional_flooded_buildings_since_prior_decade,additional_flooded_road_miles_since_prior_decade,additional_depth_based_damage_cost_since_prior_decade,additional_exposed_property_value_proxy_since_prior_decade
0,2030.0,0.50,2030 (+0.5 ft SLR),1422,153.02,"$181,009,191","$466,948,803",NaN,NaN,$0,$0
1,2040.0,1.00,2040 (+1 ft SLR),2065,183.76,"$203,879,561","$712,993,009",643.0,30.73,"$22,870,370","$246,044,206"
2,2050.0,1.50,2050 (+1.5 ft SLR),2911,217.40,"$227,241,043","$998,860,733",846.0,33.64,"$23,361,482","$285,867,724"
3,2060.0,2.00,2060 (+2 ft SLR),4173,255.45,"$263,004,538","$1,411,100,175",1262.0,38.05,"$35,763,494","$412,239,442"
4,2070.0,2.50,2070 (+2.5 ft SLR),6242,304.34,"$385,481,100","$2,062,104,014",2069.0,48.90,"$122,476,562","$651,003,839"
5,2080.0,3.00,2080 (+3 ft SLR),9148,379.33,"$510,049,254","$2,966,097,995",2906.0,74.99,"$124,568,155","$903,993,980"
6,2090.0,3.75,2090 (+3.75 ft SLR),15324,512.76,"$826,350,687","$4,957,553,138",6176.0,133.43,"$316,301,433","$1,991,455,143"
7,2100.0,4.50,2100 (+4.5 ft SLR),24256,721.89,"$1,362,343,370","$7,882,478,119",8932.0,209.13,"$535,992,683","$2,924,924,981"


In [14]:
regional_sensitivity = flood_impact_summary.sort_values("sea_level_rise_ft").copy()
regional_sensitivity["slr_interval_ft"] = regional_sensitivity["sea_level_rise_ft"].diff()
for metric in ["flooded_buildings", "flooded_road_miles", "depth_based_damage_cost", "exposed_property_value_proxy"]:
    regional_sensitivity[f"additional_{metric}"] = regional_sensitivity[metric].diff()
    regional_sensitivity[f"additional_{metric}_per_ft"] = regional_sensitivity[f"additional_{metric}"] / regional_sensitivity["slr_interval_ft"]
regional_sensitivity = regional_sensitivity[regional_sensitivity["slr_interval_ft"].notna()].copy()
regional_sensitivity["slr_interval"] = (
    "+" + (regional_sensitivity["sea_level_rise_ft"] - regional_sensitivity["slr_interval_ft"]).map(lambda value: f"{value:g}")
    + " to +" + regional_sensitivity["sea_level_rise_ft"].map(lambda value: f"{value:g}") + " ft"
)
regional_sensitivity_display = regional_sensitivity[[
    "slr_interval",
    "additional_flooded_buildings_per_ft",
    "additional_flooded_road_miles_per_ft",
    "additional_depth_based_damage_cost_per_ft",
    "additional_exposed_property_value_proxy_per_ft",
]].copy()
regional_sensitivity_display["additional_flooded_buildings_per_ft"] = regional_sensitivity_display["additional_flooded_buildings_per_ft"].round(1)
regional_sensitivity_display["additional_flooded_road_miles_per_ft"] = regional_sensitivity_display["additional_flooded_road_miles_per_ft"].round(2)
for column in ["additional_depth_based_damage_cost_per_ft", "additional_exposed_property_value_proxy_per_ft"]:
    regional_sensitivity_display[column] = regional_sensitivity_display[column].map(money)
regional_sensitivity_display


,slr_interval,additional_flooded_buildings_per_ft,additional_flooded_road_miles_per_ft,additional_depth_based_damage_cost_per_ft,additional_exposed_property_value_proxy_per_ft
1,+0 to +0.5 ft,794.0,46.33,"$34,083,204","$293,008,875"
2,+0.5 to +1 ft,1286.0,61.47,"$45,740,740","$492,088,413"
3,+1 to +1.5 ft,1692.0,67.28,"$46,722,965","$571,735,448"
4,+1.5 to +2 ft,2524.0,76.10,"$71,526,988","$824,478,884"
5,+2 to +2.5 ft,4138.0,97.80,"$244,953,124","$1,302,007,678"
6,+2.5 to +3 ft,5812.0,149.98,"$249,136,309","$1,807,987,961"
7,+3 to +3.75 ft,8234.7,177.90,"$421,735,244","$2,655,273,524"
8,+3.75 to +4 ft,10392.0,241.29,"$598,454,514","$3,399,163,087"
9,+4 to +4.5 ft,12668.0,297.62,"$772,758,108","$4,150,268,418"
10,+4.5 to +5 ft,18582.0,425.58,"$1,079,357,756","$6,170,159,213"


In [15]:
city_sensitivity = regional.sort_values(["study_area_name", "sea_level_rise_ft"]).copy()
for metric in ["flooded_building_count", "flooded_road_miles", "estimated_damage_cost", "estimated_exposed_property_value"]:
    city_sensitivity[f"additional_{metric}"] = city_sensitivity.groupby("study_area_name")[metric].diff()
city_sensitivity["slr_interval_ft"] = city_sensitivity.groupby("study_area_name")["sea_level_rise_ft"].diff()
city_sensitivity = city_sensitivity[city_sensitivity["slr_interval_ft"].notna()].copy()
city_sensitivity["additional_damage_per_ft"] = city_sensitivity["additional_estimated_damage_cost"] / city_sensitivity["slr_interval_ft"]
city_sensitivity["additional_property_value_per_ft"] = city_sensitivity["additional_estimated_exposed_property_value"] / city_sensitivity["slr_interval_ft"]
city_sensitivity["additional_buildings_per_ft"] = city_sensitivity["additional_flooded_building_count"] / city_sensitivity["slr_interval_ft"]
city_sensitivity["additional_road_miles_per_ft"] = city_sensitivity["additional_flooded_road_miles"] / city_sensitivity["slr_interval_ft"]
city_sensitivity["slr_interval"] = (
    "+" + (city_sensitivity["sea_level_rise_ft"] - city_sensitivity["slr_interval_ft"]).map(lambda value: f"{value:g}")
    + " to +" + city_sensitivity["sea_level_rise_ft"].map(lambda value: f"{value:g}") + " ft"
)
city_sensitivity_display = city_sensitivity[[
    "study_area_name",
    "slr_interval",
    "additional_buildings_per_ft",
    "additional_road_miles_per_ft",
    "additional_damage_per_ft",
    "additional_property_value_per_ft",
]].sort_values("additional_damage_per_ft", ascending=False).head(15).copy()
city_sensitivity_display["additional_buildings_per_ft"] = city_sensitivity_display["additional_buildings_per_ft"].round(1)
city_sensitivity_display["additional_road_miles_per_ft"] = city_sensitivity_display["additional_road_miles_per_ft"].round(2)
for column in ["additional_damage_per_ft", "additional_property_value_per_ft"]:
    city_sensitivity_display[column] = city_sensitivity_display[column].map(money)
city_sensitivity_display


,study_area_name,slr_interval,additional_buildings_per_ft,additional_road_miles_per_ft,additional_damage_per_ft,additional_property_value_per_ft
83,"Virginia Beach, VA",+5 to +6 ft,9628.0,128.18,"$586,166,282","$3,762,053,879"
47,"Norfolk, VA",+5 to +6 ft,3053.0,82.81,"$396,038,268","$900,905,117"
82,"Virginia Beach, VA",+4.5 to +5 ft,7896.0,158.63,"$367,713,201","$2,985,467,092"
23,"Hampton, VA",+5 to +6 ft,4768.0,77.73,"$366,320,636","$1,272,613,718"
11,"Chesapeake, VA",+5 to +6 ft,3663.0,66.65,"$280,279,626","$1,379,128,391"
81,"Virginia Beach, VA",+4 to +4.5 ft,4766.0,103.18,"$263,204,519","$1,801,317,749"
80,"Virginia Beach, VA",+3.75 to +4 ft,3860.0,84.77,"$236,550,354","$1,483,385,430"
22,"Hampton, VA",+4.5 to +5 ft,4084.0,85.50,"$231,557,720","$1,066,246,593"
46,"Norfolk, VA",+4.5 to +5 ft,2384.0,75.80,"$227,118,896","$686,689,572"
10,"Chesapeake, VA",+4.5 to +5 ft,3066.0,62.28,"$170,142,079","$1,138,774,116"


## 6. Climate-Based Flooding Possibilities

The flood rasters and exposure tables are deterministic: they estimate what is exposed if the modeled water surface occurs. They do not calculate annual flood probability. To add climate-based flooding possibilities without overclaiming, this section labels each available SLR scenario as a current-event benchmark, a future planning possibility, or a stress-test possibility.

As the heavier GIS workflow adds more scenarios, these tables will include them automatically after `scripts/export_presentation_outputs.py` refreshes the CSVs.


In [16]:
scenario_interpretation = add_climate_possibility(scenario_lookup).copy()
scenario_interpretation = scenario_interpretation[[
    "scenario_label",
    "sea_level_rise_ft",
    "planning_year",
    "scenario_type",
    "climate_possibility",
    "probability_note",
]]
scenario_interpretation

,scenario_label,sea_level_rise_ft,planning_year,scenario_type,climate_possibility,probability_note
0,Current event,0.00,NaN,current_event,Observed event benchmark,"Observed NOAA event baseline, not a recurrence..."
1,2030 (+0.5 ft SLR),0.50,2030.0,future_decade,Near-term climate possibility,"Scenario-based SLR planning case, not an annua..."
2,2040 (+1 ft SLR),1.00,2040.0,future_decade,Near-term climate possibility,"Scenario-based SLR planning case, not an annua..."
3,2050 (+1.5 ft SLR),1.50,2050.0,future_decade,Mid-century climate possibility,"Scenario-based SLR planning case, not an annua..."
4,2060 (+2 ft SLR),2.00,2060.0,future_decade,Mid-century climate possibility,"Scenario-based SLR planning case, not an annua..."
5,2070 (+2.5 ft SLR),2.50,2070.0,future_decade,Late-century high-impact possibility,"Scenario-based SLR planning case, not an annua..."
6,2080 (+3 ft SLR),3.00,2080.0,future_decade,Late-century high-impact possibility,"Scenario-based SLR planning case, not an annua..."
7,2090 (+3.75 ft SLR),3.75,2090.0,future_decade,Long-range high-impact possibility,"Scenario-based SLR planning case, not an annua..."
8,+4 ft SLR stress view,4.00,NaN,stress_view,Long-range high-impact possibility,"Sensitivity stress view, not a dated forecast"
9,2100 (+4.5 ft SLR),4.50,2100.0,future_decade,Long-range high-impact possibility,"Scenario-based SLR planning case, not an annua..."


In [17]:
risk_summary = add_climate_possibility(metric_summary).copy()
risk_summary_display = risk_summary[[
    "scenario_label",
    "climate_possibility",
    "city_count",
    "sum_flooded_building_count",
    "sum_flooded_road_miles",
    "sum_estimated_damage_cost",
    "sum_estimated_exposed_property_value",
]].rename(columns={
    "city_count": "cities_with_results",
    "sum_flooded_building_count": "flooded_buildings",
    "sum_flooded_road_miles": "flooded_road_miles",
    "sum_estimated_damage_cost": "estimated_damage_cost",
    "sum_estimated_exposed_property_value": "estimated_exposed_property_value",
})
risk_summary_display["flooded_road_miles"] = risk_summary_display["flooded_road_miles"].round(2)
for column in ["estimated_damage_cost", "estimated_exposed_property_value"]:
    risk_summary_display[column] = risk_summary_display[column].map(money)
risk_summary_display

,scenario_label,climate_possibility,cities_with_results,flooded_buildings,flooded_road_miles,estimated_damage_cost,estimated_exposed_property_value
0,Current event,Observed event benchmark,7,1025,129.86,"$163,967,589","$320,444,366"
1,2030 (+0.5 ft SLR),Near-term climate possibility,7,1422,153.02,"$181,009,191","$466,948,803"
2,2040 (+1 ft SLR),Near-term climate possibility,7,2065,183.76,"$203,879,561","$712,993,009"
3,2050 (+1.5 ft SLR),Mid-century climate possibility,7,2911,217.40,"$227,241,043","$998,860,733"
4,2060 (+2 ft SLR),Mid-century climate possibility,7,4173,255.45,"$263,004,538","$1,411,100,175"
5,2070 (+2.5 ft SLR),Late-century high-impact possibility,7,6242,304.34,"$385,481,100","$2,062,104,014"
6,2080 (+3 ft SLR),Late-century high-impact possibility,7,9148,379.33,"$510,049,254","$2,966,097,995"
7,2090 (+3.75 ft SLR),Long-range high-impact possibility,7,15324,512.76,"$826,350,687","$4,957,553,138"
8,+4 ft SLR stress view,Long-range high-impact possibility,7,17922,573.08,"$975,964,316","$5,807,343,909"
9,2100 (+4.5 ft SLR),Long-range high-impact possibility,7,24256,721.89,"$1,362,343,370","$7,882,478,119"


In [18]:
city_climate_risk = add_climate_possibility(regional).copy()
city_climate_risk = city_climate_risk.sort_values([
    "sea_level_rise_ft",
    "estimated_exposed_property_value",
    "estimated_damage_cost",
], ascending=[True, False, False])
top_city_risk = city_climate_risk.groupby("sea_level_rise_ft", as_index=False).head(3)
top_city_risk_display = top_city_risk[[
    "scenario_label",
    "climate_possibility",
    "study_area_name",
    "flooded_building_count",
    "flooded_road_miles",
    "estimated_damage_cost",
    "estimated_exposed_property_value",
]].copy()
top_city_risk_display["flooded_road_miles"] = top_city_risk_display["flooded_road_miles"].round(2)
for column in ["estimated_damage_cost", "estimated_exposed_property_value"]:
    top_city_risk_display[column] = top_city_risk_display[column].map(money)
top_city_risk_display

,scenario_label,climate_possibility,study_area_name,flooded_building_count,flooded_road_miles,estimated_damage_cost,estimated_exposed_property_value
72,Current event,Observed event benchmark,"Virginia Beach, VA",443,57.93,"$18,126,555","$158,938,510"
24,Current event,Observed event benchmark,"Newport News, VA",248,7.53,"$21,728,161","$69,073,374"
36,Current event,Observed event benchmark,"Norfolk, VA",169,14.32,"$102,698,969","$48,654,048"
73,2030 (+0.5 ft SLR),Near-term climate possibility,"Virginia Beach, VA",811,74.72,"$25,747,790","$297,927,038"
25,2030 (+0.5 ft SLR),Near-term climate possibility,"Newport News, VA",251,8.07,"$22,203,827","$69,810,397"
37,2030 (+0.5 ft SLR),Near-term climate possibility,"Norfolk, VA",175,14.91,"$103,465,127","$50,063,882"
74,2040 (+1 ft SLR),Near-term climate possibility,"Virginia Beach, VA",1381,96.42,"$36,447,274","$525,817,084"
26,2040 (+1 ft SLR),Near-term climate possibility,"Newport News, VA",257,8.87,"$27,825,580","$71,483,526"
38,2040 (+1 ft SLR),Near-term climate possibility,"Norfolk, VA",196,16.46,"$105,139,052","$54,936,418"
75,2050 (+1.5 ft SLR),Mid-century climate possibility,"Virginia Beach, VA",1958,113.66,"$52,840,497","$744,111,129"


## 7. Final +3 ft Results


In [19]:
display_columns = [
    "study_area_name",
    "flooded_building_count",
    "flooded_road_miles",
    "estimated_damage_cost",
    "median_home_value",
    "estimated_exposed_property_value",
]
plus_3ft_display = plus_3ft[display_columns].copy()
plus_3ft_display["flooded_road_miles"] = plus_3ft_display["flooded_road_miles"].round(2)
for column in ["estimated_damage_cost", "median_home_value", "estimated_exposed_property_value"]:
    plus_3ft_display[column] = plus_3ft_display[column].map(money)
plus_3ft_display

,study_area_name,flooded_building_count,flooded_road_miles,estimated_damage_cost,median_home_value,estimated_exposed_property_value
0,"Norfolk, VA",1773,53.54,"$223,492,812","$311,200","$485,979,044"
1,"Virginia Beach, VA",4464,176.38,"$153,778,430","$403,200","$1,676,290,498"
2,"Hampton, VA",1350,66.67,"$48,409,085","$266,100","$323,146,760"
3,"Chesapeake, VA",922,53.47,"$41,930,551","$388,600","$315,445,637"
4,"Newport News, VA",444,16.73,"$34,647,468","$283,200","$120,063,801"
5,"Portsmouth, VA",187,12.22,"$7,790,908","$263,900","$45,172,254"
6,"Suffolk, VA",8,0.32,$0,"$373,000",$0


## 8. Aggregate Metrics


In [20]:
row = plus_3ft_metrics.iloc[0]
aggregate_rows = [
    {"metric": "Flooded buildings", "total": row.sum_flooded_building_count, "mean": row.mean_flooded_building_count, "median": row.median_flooded_building_count, "min": row.min_flooded_building_count, "max": row.max_flooded_building_count},
    {"metric": "Flooded road count", "total": row.sum_flooded_road_count, "mean": row.mean_flooded_road_count, "median": row.median_flooded_road_count, "min": row.min_flooded_road_count, "max": row.max_flooded_road_count},
    {"metric": "Flooded road miles", "total": row.sum_flooded_road_miles, "mean": row.mean_flooded_road_miles, "median": row.median_flooded_road_miles, "min": row.min_flooded_road_miles, "max": row.max_flooded_road_miles},
    {"metric": "Estimated damage", "total": row.sum_estimated_damage_cost, "mean": row.mean_estimated_damage_cost, "median": row.median_estimated_damage_cost, "min": row.min_estimated_damage_cost, "max": row.max_estimated_damage_cost},
    {"metric": "Exposed property value", "total": row.sum_estimated_exposed_property_value, "mean": row.mean_estimated_exposed_property_value, "median": row.median_estimated_exposed_property_value, "min": row.min_estimated_exposed_property_value, "max": row.max_estimated_exposed_property_value},
]
aggregate = pd.DataFrame(aggregate_rows)
for column in ["total", "mean", "median", "min", "max"]:
    aggregate[column] = aggregate[column].astype(object)
currency_metrics = {"Estimated damage", "Exposed property value"}
for idx, item in aggregate.iterrows():
    if item["metric"] in currency_metrics:
        for column in ["total", "mean", "median", "min", "max"]:
            aggregate.loc[idx, column] = money(float(item[column]))
    else:
        for column in ["total", "mean", "median", "min", "max"]:
            aggregate.loc[idx, column] = round(float(item[column]), 2)
aggregate

,metric,total,mean,median,min,max
0,Flooded buildings,9148.0,1306.86,922.0,8.0,4464.0
1,Flooded road count,2959.0,422.71,394.0,2.0,1178.0
2,Flooded road miles,379.33,54.19,53.47,0.32,176.38
3,Estimated damage,"$510,049,254","$72,864,179","$41,930,551",$0,"$223,492,812"
4,Exposed property value,"$2,966,097,995","$423,728,285","$315,445,637",$0,"$1,676,290,498"


## 9. SLR Damage Escalation

These comparison tables summarize how impacts change from the presentation benchmark `+3 ft`, to the 2100 planning case `+4.5 ft`, to the upper `+6 ft` stress view.



In [21]:
comparison_slr_values = [3.0, 4.5, 6.0]
escalation = regional[regional["sea_level_rise_ft"].isin(comparison_slr_values)].copy()
escalation_summary = escalation.groupby("sea_level_rise_ft", as_index=False).agg(
    flooded_buildings=("flooded_building_count", "sum"),
    flooded_road_miles=("flooded_road_miles", "sum"),
    estimated_damage_cost=("estimated_damage_cost", "sum"),
    estimated_exposed_property_value=("estimated_exposed_property_value", "sum"),
    max_recovery_days=("max_estimated_recovery_days", "max"),
)
escalation_summary["scenario_label"] = escalation_summary["sea_level_rise_ft"].map(
    scenario_lookup.set_index("sea_level_rise_ft")["scenario_label"].to_dict()
)
escalation_summary = escalation_summary[[
    "scenario_label",
    "sea_level_rise_ft",
    "flooded_buildings",
    "flooded_road_miles",
    "estimated_damage_cost",
    "estimated_exposed_property_value",
    "max_recovery_days",
]]
escalation_display = escalation_summary.copy()
escalation_display["flooded_road_miles"] = escalation_display["flooded_road_miles"].round(2)
for column in ["estimated_damage_cost", "estimated_exposed_property_value"]:
    escalation_display[column] = escalation_display[column].map(money)
escalation_display


,scenario_label,sea_level_rise_ft,flooded_buildings,flooded_road_miles,estimated_damage_cost,estimated_exposed_property_value,max_recovery_days
0,2080 (+3 ft SLR),3.0,9148,379.33,"$510,049,254","$2,966,097,995",180
1,2100 (+4.5 ft SLR),4.5,24256,721.89,"$1,362,343,370","$7,882,478,119",180
2,+6 ft SLR stress view,6.0,57606,1367.24,"$3,733,834,890","$19,028,116,065",180


In [22]:
city_delta = regional[regional["sea_level_rise_ft"].isin([3.0, 6.0])].pivot_table(
    index="study_area_name",
    columns="sea_level_rise_ft",
    values=[
        "flooded_building_count",
        "flooded_road_miles",
        "estimated_damage_cost",
        "estimated_exposed_property_value",
    ],
    aggfunc="sum",
)
city_delta.columns = [f"{metric}_{slr:g}ft" for metric, slr in city_delta.columns]
city_delta = city_delta.reset_index()
city_delta["additional_flooded_buildings_3ft_to_6ft"] = city_delta["flooded_building_count_6ft"] - city_delta["flooded_building_count_3ft"]
city_delta["additional_road_miles_3ft_to_6ft"] = city_delta["flooded_road_miles_6ft"] - city_delta["flooded_road_miles_3ft"]
city_delta["additional_damage_3ft_to_6ft"] = city_delta["estimated_damage_cost_6ft"] - city_delta["estimated_damage_cost_3ft"]
city_delta["additional_property_value_3ft_to_6ft"] = city_delta["estimated_exposed_property_value_6ft"] - city_delta["estimated_exposed_property_value_3ft"]
city_delta_display = city_delta[[
    "study_area_name",
    "additional_flooded_buildings_3ft_to_6ft",
    "additional_road_miles_3ft_to_6ft",
    "additional_damage_3ft_to_6ft",
    "additional_property_value_3ft_to_6ft",
]].sort_values("additional_damage_3ft_to_6ft", ascending=False).copy()
city_delta_display["additional_road_miles_3ft_to_6ft"] = city_delta_display["additional_road_miles_3ft_to_6ft"].round(2)
for column in ["additional_damage_3ft_to_6ft", "additional_property_value_3ft_to_6ft"]:
    city_delta_display[column] = city_delta_display[column].map(money)
city_delta_display


,study_area_name,additional_flooded_buildings_3ft_to_6ft,additional_road_miles_3ft_to_6ft,additional_damage_3ft_to_6ft,additional_property_value_3ft_to_6ft
6,"Virginia Beach, VA",19333,333.55,"$1,084,133,103","$7,439,373,612"
3,"Norfolk, VA",6930,164.22,"$697,069,935","$2,023,410,838"
1,"Hampton, VA",9695,205.09,"$651,029,273","$2,515,613,025"
0,"Chesapeake, VA",7861,159.54,"$499,073,244","$2,912,922,254"
4,"Portsmouth, VA",3428,93.28,"$217,889,995","$841,403,693"
2,"Newport News, VA",1204,32.19,"$74,399,715","$326,473,669"
5,"Suffolk, VA",7,0.03,"$190,371","$2,820,979"


## 10. Scenario Trend Tables


In [23]:
damage_chart

,study_area_name,0.0,0.5,1.0,1.5,2.0,2.5,3.0,3.75,4.0,4.5,5.0,6.0
0,"Chesapeake, VA",9.547998e+06,1.716860e+07,1.801137e+07,1.880791e+07,2.112847e+07,2.865725e+07,4.193055e+07,8.827400e+07,1.084067e+08,1.756531e+08,2.607242e+08,5.410038e+08
1,"Hampton, VA",7.565402e+06,8.071565e+06,1.190420e+07,1.384465e+07,1.862928e+07,2.997807e+07,4.840909e+07,1.054796e+08,1.372911e+08,2.173389e+08,3.331177e+08,6.994384e+08
2,"Newport News, VA",2.172816e+07,2.220383e+07,2.782558e+07,2.839919e+07,2.931234e+07,3.148873e+07,3.464747e+07,3.980050e+07,4.215759e+07,4.986696e+07,6.197381e+07,1.090472e+08
3,"Norfolk, VA",1.026990e+08,1.034651e+08,1.051391e+08,1.086274e+08,1.152168e+08,1.836132e+08,2.234928e+08,2.970760e+08,3.279284e+08,4.109650e+08,5.245245e+08,9.205627e+08
4,"Portsmouth, VA",4.300504e+06,4.352282e+06,4.552085e+06,4.721429e+06,4.969675e+06,6.858774e+06,7.790908e+06,1.856760e+07,2.388856e+07,4.060206e+07,6.986184e+07,2.256809e+08
5,"Suffolk, VA",0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,4.169419e+03,5.625282e+03,2.868217e+04,7.497930e+04,1.903714e+05
6,"Virginia Beach, VA",1.812655e+07,2.574779e+07,3.644727e+07,5.284050e+07,7.374801e+07,1.048851e+08,1.537784e+08,2.771488e+08,3.362864e+08,4.678887e+08,6.517453e+08,1.237912e+09


In [24]:
roads_chart

,study_area_name,0.0,0.5,1.0,1.5,2.0,2.5,3.0,3.75,4.0,4.5,5.0,6.0
0,"Chesapeake, VA",11.120616,14.470480,18.853331,24.915452,31.515314,41.800471,53.467131,74.841783,86.235256,115.222781,146.362656,213.010234
1,"Hampton, VA",30.050791,31.747078,33.811537,39.643114,47.067606,54.876751,66.669255,98.252220,114.504861,151.279468,194.028553,271.760658
2,"Newport News, VA",7.527600,8.074054,8.871619,9.766995,12.315322,14.564616,16.733148,21.420246,23.406687,27.787397,36.862274,48.919194
3,"Norfolk, VA",14.317389,14.909575,16.463100,19.674482,24.254974,32.248622,53.538556,70.520136,77.150869,97.048870,134.948289,217.760677
4,"Portsmouth, VA",8.582277,8.775485,9.011045,9.416784,10.004031,10.806144,12.217257,17.744813,20.609974,27.790878,40.378900,105.495160
5,"Suffolk, VA",0.323073,0.323073,0.323073,0.323073,0.323073,0.323073,0.323073,0.323073,0.323073,0.323073,0.343388,0.356025
6,"Virginia Beach, VA",57.933872,74.722925,96.423781,113.656141,129.964841,149.724131,176.384302,229.655975,250.849640,302.438517,381.754807,509.934697


In [25]:
buildings_chart

,study_area_name,0.0,0.5,1.0,1.5,2.0,2.5,3.0,3.75,4.0,4.5,5.0,6.0
0,"Chesapeake, VA",55,65,83,133,247,516,922,1849,2347,3587,5120,8783
1,"Hampton, VA",50,56,78,171,389,794,1350,2425,2966,4235,6277,11045
2,"Newport News, VA",248,251,257,268,304,346,444,605,659,786,968,1648
3,"Norfolk, VA",169,175,196,306,512,1009,1773,3011,3462,4458,5650,8703
4,"Portsmouth, VA",59,63,68,73,84,123,187,560,648,967,1361,3615
5,"Suffolk, VA",1,1,2,2,4,6,8,1,2,2,2,15
6,"Virginia Beach, VA",443,811,1381,1958,2633,3448,4464,6873,7838,10221,14169,23797


In [26]:
property_chart

,study_area_name,0.0,0.5,1.0,1.5,2.0,2.5,3.0,3.75,4.0,4.5,5.0,6.0
0,"Chesapeake, VA",1.777859e+07,2.114053e+07,2.788320e+07,4.457743e+07,8.367167e+07,1.760465e+08,3.154456e+08,6.462621e+08,8.230365e+08,1.279852e+09,1.849240e+09,3.228368e+09
1,"Hampton, VA",1.100391e+07,1.200325e+07,1.558920e+07,3.751381e+07,8.905354e+07,1.881242e+08,3.231468e+08,5.876830e+08,7.198665e+08,1.033023e+09,1.566146e+09,2.838760e+09
2,"Newport News, VA",6.907337e+07,6.981040e+07,7.148353e+07,7.383112e+07,8.224371e+07,9.460612e+07,1.200638e+08,1.621467e+08,1.778650e+08,2.117087e+08,2.621959e+08,4.465375e+08
3,"Norfolk, VA",4.865405e+07,5.006388e+07,5.493642e+07,8.037110e+07,1.361465e+08,2.721543e+08,4.859790e+08,8.383117e+08,9.704522e+08,1.265140e+09,1.608485e+09,2.509390e+09
4,"Portsmouth, VA",1.499594e+07,1.600371e+07,1.728358e+07,1.845614e+07,2.106561e+07,2.955303e+07,4.517225e+07,1.336852e+08,1.557805e+08,2.315175e+08,3.274639e+08,8.865759e+08
5,"Suffolk, VA",0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,9.298922e+04,1.253772e+05,3.600480e+05,4.173861e+05,2.820979e+06
6,"Virginia Beach, VA",1.589385e+08,2.979270e+08,5.258171e+08,7.441111e+08,9.989191e+08,1.301620e+09,1.676290e+09,2.589371e+09,2.960218e+09,3.860877e+09,5.353610e+09,9.115664e+09


## 11. Damage Calculations

This section reads feature-level damage estimates directly from PostGIS and rolls them up by city, SLR scenario, damage class, and recovery class. The existing depth-based damage estimate is kept as a screening structural-damage metric. For non-current SLR scenarios, including the `+4`, `+5`, and `+6 ft` flood-on-top-of-SLR stress views, the notebook also reports a total-loss property assumption using ACS median home value times the number of damaged buildings.


In [27]:
engine = get_engine()
damage_breakdown_query = text(
    """
    SELECT
        d.study_area_id,
        sa.name AS study_area_name,
        d.scenario_id,
        s.sea_level_rise_ft,
        d.damage_class,
        d.recovery_class,
        count(*) AS damaged_building_count,
        sum(d.estimated_structure_value) AS estimated_structure_value,
        sum(d.estimated_damage_cost) AS depth_based_damage_cost,
        avg(d.estimated_damage_cost) AS average_depth_based_damage_cost,
        max(d.max_depth_ft) AS max_depth_ft,
        max(d.estimated_recovery_days) AS max_estimated_recovery_days,
        max(ps.median_home_value) AS median_home_value
    FROM results.connected_building_damage_estimates d
    JOIN processed.study_areas sa
      ON sa.study_area_id = d.study_area_id
    JOIN processed.flood_scenarios s
      ON s.scenario_id = d.scenario_id
    LEFT JOIN results.property_value_exposure_summary ps
      ON ps.study_area_id = d.study_area_id
     AND ps.scenario_id = d.scenario_id
     AND ps.acs_year = :acs_year
    WHERE d.study_area_id = ANY(:study_area_ids)
    GROUP BY d.study_area_id, sa.name, d.scenario_id, s.sea_level_rise_ft, d.damage_class, d.recovery_class
    ORDER BY s.sea_level_rise_ft, sa.name, d.damage_class, d.recovery_class
    """
)
damage_breakdown = pd.read_sql(
    damage_breakdown_query,
    engine,
    params={"acs_year": ACS_YEAR, "study_area_ids": list(CITY_EXPORTS)},
)
slr_to_decade = {v: k for k, v in FUTURE_SLR_DECADES.items()}
damage_breakdown["planning_year"] = damage_breakdown["sea_level_rise_ft"].map(slr_to_decade).astype("Int64")
damage_breakdown["scenario_type"] = damage_breakdown["sea_level_rise_ft"].map(
    lambda value: "current_event" if value == 0 else "future_decade" if value in slr_to_decade else "stress_view"
)
damage_breakdown["scenario_label"] = damage_breakdown.apply(
    lambda row: "Current event"
    if row.sea_level_rise_ft == 0
    else f"{int(row.planning_year)} (+{row.sea_level_rise_ft:g} ft SLR)"
    if pd.notna(row.planning_year)
    else f"+{row.sea_level_rise_ft:g} ft SLR stress view",
    axis=1,
)
damage_breakdown["climate_total_loss_property_value"] = damage_breakdown["damaged_building_count"] * damage_breakdown["median_home_value"].fillna(0)
damage_breakdown.loc[damage_breakdown["sea_level_rise_ft"] == 0, "climate_total_loss_property_value"] = 0
damage_breakdown["loss_assumption"] = damage_breakdown["sea_level_rise_ft"].map(
    lambda value: "depth-based damage only" if value == 0 else "total property loss for SLR/flood stress"
)
print(f"damage_breakdown_rows={len(damage_breakdown)}")

damage_breakdown_rows=278


In [28]:
damage_class_display = damage_breakdown[
    [
        "study_area_name",
        "scenario_label",
        "damage_class",
        "recovery_class",
        "damaged_building_count",
        "depth_based_damage_cost",
        "climate_total_loss_property_value",
        "max_depth_ft",
        "max_estimated_recovery_days",
        "loss_assumption",
    ]
].copy()
damage_class_display["max_depth_ft"] = damage_class_display["max_depth_ft"].round(2)
format_money_columns(damage_class_display, ["depth_based_damage_cost", "climate_total_loss_property_value"])

,study_area_name,scenario_label,damage_class,recovery_class,damaged_building_count,depth_based_damage_cost,climate_total_loss_property_value,max_depth_ft,max_estimated_recovery_days,loss_assumption
0,"Chesapeake, VA",Current event,major,months,17,"$2,096,900",$0,3.73,90,depth-based damage only
1,"Chesapeake, VA",Current event,minor,days,8,"$50,733",$0,0.98,7,depth-based damage only
2,"Chesapeake, VA",Current event,moderate,weeks,30,"$7,400,364",$0,2.60,30,depth-based damage only
3,"Hampton, VA",Current event,major,months,30,"$6,841,935",$0,5.14,90,depth-based damage only
4,"Hampton, VA",Current event,minor,days,7,"$58,978",$0,0.70,7,depth-based damage only
...,...,...,...,...,...,...,...,...,...,...
273,"Suffolk, VA",+6 ft SLR stress view,moderate,weeks,2,"$95,048","$746,000",2.58,30,total property loss for SLR/flood stress
274,"Virginia Beach, VA",+6 ft SLR stress view,major,months,4729,"$505,745,814","$1,906,732,800",6.00,90,total property loss for SLR/flood stress
275,"Virginia Beach, VA",+6 ft SLR stress view,minor,days,8958,"$101,816,692","$3,611,865,600",1.00,7,total property loss for SLR/flood stress
276,"Virginia Beach, VA",+6 ft SLR stress view,moderate,weeks,9589,"$554,667,184","$3,866,284,800",3.00,30,total property loss for SLR/flood stress


In [29]:
damage_loss_summary = damage_breakdown.groupby(
    ["sea_level_rise_ft", "scenario_label", "scenario_type"], as_index=False
).agg(
    damaged_buildings=("damaged_building_count", "sum"),
    depth_based_damage_cost=("depth_based_damage_cost", "sum"),
    climate_total_loss_property_value=("climate_total_loss_property_value", "sum"),
    max_depth_ft=("max_depth_ft", "max"),
    max_estimated_recovery_days=("max_estimated_recovery_days", "max"),
)
damage_loss_summary["max_depth_ft"] = damage_loss_summary["max_depth_ft"].round(2)
format_money_columns(damage_loss_summary, ["depth_based_damage_cost", "climate_total_loss_property_value"])

,sea_level_rise_ft,scenario_label,scenario_type,damaged_buildings,depth_based_damage_cost,climate_total_loss_property_value,max_depth_ft,max_estimated_recovery_days
0,0.00,Current event,current_event,1024,"$163,967,589",$0,47.45,180
1,0.50,2030 (+0.5 ft SLR),future_decade,1421,"$181,009,191","$509,324,700",47.95,180
2,1.00,2040 (+1 ft SLR),future_decade,2063,"$203,879,561","$761,551,600",48.45,180
3,1.50,2050 (+1.5 ft SLR),future_decade,2909,"$227,241,043","$1,077,042,000",48.95,180
4,2.00,2060 (+2 ft SLR),future_decade,4169,"$263,004,538","$1,528,717,500",49.45,180
5,2.50,2070 (+2.5 ft SLR),future_decade,6236,"$385,481,100","$2,246,482,300",49.95,180
6,3.00,2080 (+3 ft SLR),future_decade,9140,"$510,049,254","$3,244,256,700",50.45,180
7,3.75,2090 (+3.75 ft SLR),future_decade,15324,"$826,350,687","$5,391,523,700",51.20,180
8,4.00,+4 ft SLR stress view,stress_view,17922,"$975,964,316","$6,297,334,800",51.45,180
9,4.50,2100 (+4.5 ft SLR),future_decade,24256,"$1,362,343,370","$8,507,811,000",51.95,180


In [30]:
stress_damage = damage_breakdown[damage_breakdown["sea_level_rise_ft"] >= 4.0].copy()
if stress_damage.empty:
    print("No +4 ft or higher stress rows available yet. Enable RUN_FULL_0_TO_6FT_WORKFLOW and rerun the notebook to generate them.")
else:
    stress_damage_display = stress_damage.groupby(
        ["study_area_name", "sea_level_rise_ft", "scenario_label"], as_index=False
    ).agg(
        damaged_buildings=("damaged_building_count", "sum"),
        depth_based_damage_cost=("depth_based_damage_cost", "sum"),
        climate_total_loss_property_value=("climate_total_loss_property_value", "sum"),
        max_depth_ft=("max_depth_ft", "max"),
    ).sort_values(["sea_level_rise_ft", "climate_total_loss_property_value"], ascending=[True, False])
    stress_damage_display["max_depth_ft"] = stress_damage_display["max_depth_ft"].round(2)
    display(format_money_columns(stress_damage_display, ["depth_based_damage_cost", "climate_total_loss_property_value"]))

,study_area_name,sea_level_rise_ft,scenario_label,damaged_buildings,depth_based_damage_cost,climate_total_loss_property_value,max_depth_ft
24,"Virginia Beach, VA",4.0,+4 ft SLR stress view,7838,"$336,286,391","$3,160,281,600",8.00
12,"Norfolk, VA",4.0,+4 ft SLR stress view,3462,"$327,928,397","$1,077,374,400",8.22
0,"Chesapeake, VA",4.0,+4 ft SLR stress view,2347,"$108,406,652","$912,044,200",7.73
4,"Hampton, VA",4.0,+4 ft SLR stress view,2966,"$137,291,098","$789,252,600",17.32
8,"Newport News, VA",4.0,+4 ft SLR stress view,659,"$42,157,590","$186,628,800",42.73
16,"Portsmouth, VA",4.0,+4 ft SLR stress view,648,"$23,888,563","$171,007,200",51.45
20,"Suffolk, VA",4.0,+4 ft SLR stress view,2,"$5,625","$746,000",0.58
25,"Virginia Beach, VA",4.5,2100 (+4.5 ft SLR),10221,"$467,888,651","$4,121,107,200",8.50
1,"Chesapeake, VA",4.5,2100 (+4.5 ft SLR),3587,"$175,653,129","$1,393,908,200",8.23
13,"Norfolk, VA",4.5,2100 (+4.5 ft SLR),4458,"$410,965,031","$1,387,329,600",8.72


## 12. Top Impacted Roads


In [31]:
top_roads_plus_3ft = top_roads[top_roads["sea_level_rise_ft"] == 3.0].copy()
top_roads_plus_3ft["flooded_length_mi"] = top_roads_plus_3ft["flooded_length_mi"].round(3)
top_roads_plus_3ft[["study_area_name", "rank", "road_name", "mtfcc", "flooded_length_mi", "max_depth_ft"]].head(30)

,study_area_name,rank,road_name,mtfcc,flooded_length_mi,max_depth_ft
60,"Chesapeake, VA",1,Mount Pleasant Rd,S1200,1.703,6.726153
61,"Chesapeake, VA",2,State Rte 165,S1200,1.703,6.726153
62,"Chesapeake, VA",3,Indian Creek Rd,S1400,1.414,6.726153
63,"Chesapeake, VA",4,Bunch Walnuts Rd,S1400,1.406,6.726153
64,"Chesapeake, VA",5,(unnamed road),S1400,1.146,6.726153
65,"Chesapeake, VA",6,Lake Drummond Cswy,S1400,1.058,6.726153
66,"Chesapeake, VA",7,Blackwater Rd,S1400,1.036,6.726153
67,"Chesapeake, VA",8,(unnamed road),S1400,1.027,6.726153
68,"Chesapeake, VA",9,(unnamed road),S1400,1.008,6.726153
69,"Chesapeake, VA",10,(unnamed road),S1400,0.871,6.726153


## Priority Transportation Corridors

General road metrics summarize all flooded road mileage. This section separately tracks recognizable regional corridors and mobility chokepoints, including interstates, US/state routes, and named local roads such as Battlefield Blvd and Princess Anne Rd. These are screening-level flooded-mile estimates from TIGER road centerlines and connected flood extents, not closure predictions.


In [32]:
priority_corridor_query = text(
    """
    WITH priority_patterns(corridor_family, corridor_name, pattern) AS (
        VALUES
            ('Interstate', 'I-64', 'i- 64'),
            ('Interstate', 'I-264', 'i- 264'),
            ('Interstate', 'I-464', 'i- 464'),
            ('Interstate', 'I-664', 'i- 664'),
            ('US / State Route', 'US Hwy 13', 'us hwy 13'),
            ('US / State Route', 'US Hwy 17', 'us hwy 17'),
            ('US / State Route', 'US Hwy 58', 'us hwy 58'),
            ('US / State Route', 'US Hwy 60', 'us hwy 60'),
            ('US / State Route', 'US Hwy 460', 'us hwy 460'),
            ('US / State Route', 'State Rte 168', 'state rte 168'),
            ('US / State Route', 'State Rte 337', 'state rte 337'),
            ('Named Local / Regional', 'Battlefield Blvd', 'battlefield'),
            ('Named Local / Regional', 'Princess Anne Rd', 'princess anne'),
            ('Named Local / Regional', 'Shore Dr', 'shore dr'),
            ('Named Local / Regional', 'Military Hwy', 'military hwy'),
            ('Named Local / Regional', 'Virginia Beach Blvd', 'virginia beach blvd'),
            ('Named Local / Regional', 'Tidewater Dr', 'tidewater dr'),
            ('Named Local / Regional', 'Independence Blvd', 'independence blvd'),
            ('Named Local / Regional', 'Hampton Roads Bridge-Tunnel', 'hampton roads bridge tunl')
    )
    SELECT
        pp.corridor_family,
        pp.corridor_name,
        sa.study_area_id,
        sa.name AS study_area_name,
        s.scenario_id,
        s.sea_level_rise_ft,
        sum(i.flooded_length_mi) AS flooded_length_mi,
        max(i.max_depth_ft) AS max_depth_ft,
        count(DISTINCT i.road_id) AS segment_count
    FROM priority_patterns pp
    JOIN results.connected_road_flood_impacts i
      ON lower(coalesce(i.fullname, '')) LIKE '%' || pp.pattern || '%'
    JOIN processed.study_areas sa
      ON sa.study_area_id = i.study_area_id
    JOIN processed.flood_scenarios s
      ON s.scenario_id = i.scenario_id
    WHERE i.study_area_id = ANY(:study_area_ids)
    GROUP BY pp.corridor_family, pp.corridor_name, sa.study_area_id, sa.name, s.scenario_id, s.sea_level_rise_ft
    ORDER BY pp.corridor_family, pp.corridor_name, sa.name, s.sea_level_rise_ft
    """
)
priority_corridors = pd.read_sql(
    priority_corridor_query,
    engine,
    params={"study_area_ids": list(CITY_EXPORTS)},
)
priority_corridors = priority_corridors.merge(scenario_lookup, on="sea_level_rise_ft", how="left")
priority_corridors = add_climate_possibility(priority_corridors)
print(f"priority_corridor_rows={len(priority_corridors)}")
print(f"priority_corridors={priority_corridors['corridor_name'].nunique()}")


priority_corridor_rows=575
priority_corridors=19


In [33]:
interstate_exposure = priority_corridors[priority_corridors["corridor_family"] == "Interstate"].copy()
interstate_summary = interstate_exposure.groupby(["sea_level_rise_ft", "scenario_label", "corridor_name"], as_index=False).agg(
    flooded_length_mi=("flooded_length_mi", "sum"),
    max_depth_ft=("max_depth_ft", "max"),
    segment_count=("segment_count", "sum"),
)
interstate_pivot = interstate_summary.pivot_table(
    index=["sea_level_rise_ft", "scenario_label"],
    columns="corridor_name",
    values="flooded_length_mi",
    aggfunc="sum",
    fill_value=0,
).reset_index()
interstate_columns = [column for column in interstate_pivot.columns if column not in {"sea_level_rise_ft", "scenario_label"}]
interstate_pivot["total_interstate_flooded_miles"] = interstate_pivot[interstate_columns].sum(axis=1)
for column in interstate_columns + ["total_interstate_flooded_miles"]:
    interstate_pivot[column] = interstate_pivot[column].round(2)
interstate_pivot


corridor_name,sea_level_rise_ft,scenario_label,I-264,I-464,I-64,I-664,total_interstate_flooded_miles
0,0.00,Current event,1.91,0.57,9.97,4.27,16.73
1,0.50,2030 (+0.5 ft SLR),1.92,0.60,9.99,4.39,16.89
2,1.00,2040 (+1 ft SLR),1.93,0.60,10.02,4.49,17.04
3,1.50,2050 (+1.5 ft SLR),1.93,0.61,10.05,4.54,17.12
4,2.00,2060 (+2 ft SLR),1.93,0.62,10.07,4.63,17.25
5,2.50,2070 (+2.5 ft SLR),1.94,0.64,10.10,4.71,17.39
6,3.00,2080 (+3 ft SLR),1.99,0.70,10.25,4.76,17.70
7,3.75,2090 (+3.75 ft SLR),2.11,0.72,10.65,4.86,18.33
8,4.00,+4 ft SLR stress view,2.22,0.72,10.69,4.93,18.55
9,4.50,2100 (+4.5 ft SLR),2.32,1.29,10.94,5.11,19.67


In [34]:
key_scenarios = [3.0, 4.5, 6.0]
priority_key = priority_corridors[priority_corridors["sea_level_rise_ft"].isin(key_scenarios)].copy()
priority_key_display = priority_key[[
    "corridor_family",
    "corridor_name",
    "study_area_name",
    "scenario_label",
    "flooded_length_mi",
    "max_depth_ft",
    "segment_count",
]].sort_values(["scenario_label", "flooded_length_mi"], ascending=[True, False]).copy()
priority_key_display["flooded_length_mi"] = priority_key_display["flooded_length_mi"].round(2)
priority_key_display["max_depth_ft"] = priority_key_display["max_depth_ft"].round(2)
priority_key_display.head(50)


,corridor_family,corridor_name,study_area_name,scenario_label,flooded_length_mi,max_depth_ft,segment_count
406,US / State Route,US Hwy 13,"Virginia Beach, VA",+6 ft SLR stress view,14.55,33.06,8
358,US / State Route,State Rte 337,"Norfolk, VA",+6 ft SLR stress view,8.49,29.21,10
262,Named Local / Regional,Shore Dr,"Virginia Beach, VA",+6 ft SLR stress view,8.38,33.06,9
574,US / State Route,US Hwy 60,"Virginia Beach, VA",+6 ft SLR stress view,8.10,33.06,5
538,US / State Route,US Hwy 60,"Hampton, VA",+6 ft SLR stress view,7.66,19.39,5
562,US / State Route,US Hwy 60,"Norfolk, VA",+6 ft SLR stress view,7.51,29.21,6
334,US / State Route,State Rte 168,"Norfolk, VA",+6 ft SLR stress view,6.76,29.21,9
83,Interstate,I-64,"Hampton, VA",+6 ft SLR stress view,6.70,19.39,2
322,US / State Route,State Rte 168,"Hampton, VA",+6 ft SLR stress view,6.70,19.39,2
466,US / State Route,US Hwy 460,"Norfolk, VA",+6 ft SLR stress view,6.10,29.21,6


In [35]:
corridor_escalation = priority_corridors[priority_corridors["sea_level_rise_ft"].isin([3.0, 6.0])].pivot_table(
    index=["corridor_family", "corridor_name", "study_area_name"],
    columns="sea_level_rise_ft",
    values="flooded_length_mi",
    aggfunc="sum",
    fill_value=0,
).reset_index()
if 3.0 not in corridor_escalation.columns:
    corridor_escalation[3.0] = 0.0
if 6.0 not in corridor_escalation.columns:
    corridor_escalation[6.0] = 0.0
corridor_escalation = corridor_escalation.rename(columns={3.0: "plus_3ft_miles", 6.0: "plus_6ft_miles"})
corridor_escalation["additional_miles_3ft_to_6ft"] = corridor_escalation["plus_6ft_miles"] - corridor_escalation["plus_3ft_miles"]
corridor_escalation["percent_increase_3ft_to_6ft"] = corridor_escalation.apply(
    lambda row: None if row["plus_3ft_miles"] == 0 else row["additional_miles_3ft_to_6ft"] / row["plus_3ft_miles"] * 100,
    axis=1,
)
corridor_escalation_display = corridor_escalation.sort_values("plus_6ft_miles", ascending=False).head(40).copy()
for column in ["plus_3ft_miles", "plus_6ft_miles", "additional_miles_3ft_to_6ft"]:
    corridor_escalation_display[column] = corridor_escalation_display[column].round(2)
corridor_escalation_display["percent_increase_3ft_to_6ft"] = corridor_escalation_display["percent_increase_3ft_to_6ft"].round(1)
corridor_escalation_display


sea_level_rise_ft,corridor_family,corridor_name,study_area_name,plus_3ft_miles,plus_6ft_miles,additional_miles_3ft_to_6ft,percent_increase_3ft_to_6ft
37,US / State Route,US Hwy 13,"Virginia Beach, VA",14.27,14.55,0.27,1.9
33,US / State Route,State Rte 337,"Norfolk, VA",2.72,8.49,5.76,211.7
25,Named Local / Regional,Shore Dr,"Virginia Beach, VA",0.96,8.38,7.42,772.5
51,US / State Route,US Hwy 60,"Virginia Beach, VA",1.03,8.10,7.07,684.3
48,US / State Route,US Hwy 60,"Hampton, VA",6.19,7.66,1.47,23.8
50,US / State Route,US Hwy 60,"Norfolk, VA",3.76,7.51,3.74,99.4
31,US / State Route,State Rte 168,"Norfolk, VA",4.33,6.76,2.43,56.2
30,US / State Route,State Rte 168,"Hampton, VA",6.30,6.70,0.40,6.4
6,Interstate,I-64,"Hampton, VA",6.30,6.70,0.40,6.4
42,US / State Route,US Hwy 460,"Norfolk, VA",2.06,6.10,4.04,196.1


In [36]:
priority_sensitivity = priority_corridors.sort_values(["corridor_name", "study_area_name", "sea_level_rise_ft"]).copy()
priority_sensitivity["slr_interval_ft"] = priority_sensitivity.groupby(["corridor_name", "study_area_name"])["sea_level_rise_ft"].diff()
priority_sensitivity["additional_flooded_miles"] = priority_sensitivity.groupby(["corridor_name", "study_area_name"])["flooded_length_mi"].diff()
priority_sensitivity = priority_sensitivity[priority_sensitivity["slr_interval_ft"].notna()].copy()
priority_sensitivity["additional_flooded_miles_per_ft"] = priority_sensitivity["additional_flooded_miles"] / priority_sensitivity["slr_interval_ft"]
priority_sensitivity["slr_interval"] = (
    "+" + (priority_sensitivity["sea_level_rise_ft"] - priority_sensitivity["slr_interval_ft"]).map(lambda value: f"{value:g}")
    + " to +" + priority_sensitivity["sea_level_rise_ft"].map(lambda value: f"{value:g}") + " ft"
)
priority_sensitivity_display = priority_sensitivity[[
    "corridor_family",
    "corridor_name",
    "study_area_name",
    "slr_interval",
    "additional_flooded_miles_per_ft",
    "flooded_length_mi",
]].sort_values("additional_flooded_miles_per_ft", ascending=False).head(30).copy()
priority_sensitivity_display["additional_flooded_miles_per_ft"] = priority_sensitivity_display["additional_flooded_miles_per_ft"].round(2)
priority_sensitivity_display["flooded_length_mi"] = priority_sensitivity_display["flooded_length_mi"].round(2)
priority_sensitivity_display


,corridor_family,corridor_name,study_area_name,slr_interval,additional_flooded_miles_per_ft,flooded_length_mi
329,US / State Route,State Rte 168,"Norfolk, VA",+2.5 to +3 ft,6.17,4.33
557,US / State Route,US Hwy 60,"Norfolk, VA",+2.5 to +3 ft,6.08,3.76
357,US / State Route,State Rte 337,"Norfolk, VA",+4.5 to +5 ft,4.63,5.94
261,Named Local / Regional,Shore Dr,"Virginia Beach, VA",+4.5 to +5 ft,4.51,5.70
573,US / State Route,US Hwy 60,"Virginia Beach, VA",+4.5 to +5 ft,4.46,5.45
465,US / State Route,US Hwy 460,"Norfolk, VA",+4.5 to +5 ft,4.19,4.68
259,Named Local / Regional,Shore Dr,"Virginia Beach, VA",+3.75 to +4 ft,3.02,2.67
262,Named Local / Regional,Shore Dr,"Virginia Beach, VA",+5 to +6 ft,2.67,8.38
574,US / State Route,US Hwy 60,"Virginia Beach, VA",+5 to +6 ft,2.66,8.10
358,US / State Route,State Rte 337,"Norfolk, VA",+5 to +6 ft,2.55,8.49


Priority corridor caveat: TIGER road lines are generalized centerlines, and bridges/tunnels may intersect low-elevation raster cells or nearby connected flood extents. Use these results as a screening view of flooded corridor mileage and sensitivity, not as an engineering closure forecast.


## 13. Future Decade Results

These tables appear after the optional future SLR workflow has populated the `+4.5 ft` 2100 scenario. They use the same connected-inundation, exposure, damage, and ACS property-value model as the current `+3 ft` reports.


In [37]:
slr_to_decade = {v: k for k, v in FUTURE_SLR_DECADES.items()}
future_regional = regional[regional["sea_level_rise_ft"].isin(slr_to_decade)].copy()
future_regional["decade"] = future_regional["sea_level_rise_ft"].map(slr_to_decade).astype("Int64")
available_decades = sorted(future_regional["decade"].dropna().unique().tolist())
print(f"available_future_decades={available_decades}")
if 2100 not in available_decades:
    print("2100 (+4.5 ft) is not available yet. Enable RUN_FULL_0_TO_6FT_WORKFLOW and rerun the notebook to generate it.")

available_future_decades=[2030, 2040, 2050, 2060, 2070, 2080, 2090, 2100]


In [38]:
future_decade_summary = future_regional.groupby(["decade", "sea_level_rise_ft"], as_index=False).agg(
    flooded_buildings=("flooded_building_count", "sum"),
    flooded_road_miles=("flooded_road_miles", "sum"),
    estimated_damage_cost=("estimated_damage_cost", "sum"),
    estimated_exposed_property_value=("estimated_exposed_property_value", "sum"),
)
future_decade_display = future_decade_summary.copy()
future_decade_display["flooded_road_miles"] = future_decade_display["flooded_road_miles"].round(2)
for column in ["estimated_damage_cost", "estimated_exposed_property_value"]:
    future_decade_display[column] = future_decade_display[column].map(money)
future_decade_display

,decade,sea_level_rise_ft,flooded_buildings,flooded_road_miles,estimated_damage_cost,estimated_exposed_property_value
0,2030,0.50,1422,153.02,"$181,009,191","$466,948,803"
1,2040,1.00,2065,183.76,"$203,879,561","$712,993,009"
2,2050,1.50,2911,217.40,"$227,241,043","$998,860,733"
3,2060,2.00,4173,255.45,"$263,004,538","$1,411,100,175"
4,2070,2.50,6242,304.34,"$385,481,100","$2,062,104,014"
5,2080,3.00,9148,379.33,"$510,049,254","$2,966,097,995"
6,2090,3.75,15324,512.76,"$826,350,687","$4,957,553,138"
7,2100,4.50,24256,721.89,"$1,362,343,370","$7,882,478,119"


In [39]:
future_2100 = future_regional[future_regional["decade"] == 2100].copy()
if future_2100.empty:
    print("No 2100 (+4.5 ft) rows available yet.")
else:
    future_2100 = future_2100.sort_values("estimated_damage_cost", ascending=False)
    future_2100_display = future_2100[display_columns].copy()
    future_2100_display["flooded_road_miles"] = future_2100_display["flooded_road_miles"].round(2)
    for column in ["estimated_damage_cost", "median_home_value", "estimated_exposed_property_value"]:
        future_2100_display[column] = future_2100_display[column].map(money)
    display(future_2100_display)

,study_area_name,flooded_building_count,flooded_road_miles,estimated_damage_cost,median_home_value,estimated_exposed_property_value
81,"Virginia Beach, VA",10221,302.44,"$467,888,651","$403,200","$3,860,876,685"
45,"Norfolk, VA",4458,97.05,"$410,965,031","$311,200","$1,265,139,979"
21,"Hampton, VA",4235,151.28,"$217,338,862","$266,100","$1,033,022,770"
9,"Chesapeake, VA",3587,115.22,"$175,653,129","$388,600","$1,279,852,443"
33,"Newport News, VA",786,27.79,"$49,866,958","$283,200","$211,708,658"
57,"Portsmouth, VA",967,27.79,"$40,602,057","$263,900","$231,517,535"
69,"Suffolk, VA",2,0.32,"$28,682","$373,000","$360,048"


## 14. +6 ft Stress View

This table shows the upper flood-view layer requested for flooding on top of existing sea-level-rise scenarios. It is a stress view, not a dated projection.


In [40]:
if plus_6ft.empty:
    print("No +6 ft rows available yet. Enable RUN_FULL_0_TO_6FT_WORKFLOW and rerun the notebook to generate them.")
else:
    plus_6ft_display = plus_6ft.sort_values("estimated_damage_cost", ascending=False)[display_columns].copy()
    plus_6ft_display["flooded_road_miles"] = plus_6ft_display["flooded_road_miles"].round(2)
    for column in ["estimated_damage_cost", "median_home_value", "estimated_exposed_property_value"]:
        plus_6ft_display[column] = plus_6ft_display[column].map(money)
    display(plus_6ft_display)

,study_area_name,flooded_building_count,flooded_road_miles,estimated_damage_cost,median_home_value,estimated_exposed_property_value
0,"Virginia Beach, VA",23797,509.93,"$1,237,911,533","$403,200","$9,115,664,110"
1,"Norfolk, VA",8703,217.76,"$920,562,748","$311,200","$2,509,389,882"
2,"Hampton, VA",11045,271.76,"$699,438,358","$266,100","$2,838,759,785"
3,"Chesapeake, VA",8783,213.01,"$541,003,795","$388,600","$3,228,367,892"
4,"Portsmouth, VA",3615,105.50,"$225,680,903","$263,900","$886,575,947"
5,"Newport News, VA",1648,48.92,"$109,047,183","$283,200","$446,537,470"
6,"Suffolk, VA",15,0.36,"$190,371","$373,000","$2,820,979"


## 15. Report Files


In [41]:
for path in sorted(REPORT_DIR.glob("regional*.csv")):
    print(path.relative_to(PROJECT_ROOT))

data/processed/gis/regional_2100_metric_summary.csv
data/processed/gis/regional_chart_2100_summary.csv
data/processed/gis/regional_chart_all_slr_summary.csv
data/processed/gis/regional_chart_damage_by_city_scenario.csv
data/processed/gis/regional_chart_decade_summary.csv
data/processed/gis/regional_chart_exposed_property_value_by_city_scenario.csv
data/processed/gis/regional_chart_flooded_buildings_by_city_scenario.csv
data/processed/gis/regional_chart_flooded_road_miles_by_city_scenario.csv
data/processed/gis/regional_chart_plus_3ft_summary.csv
data/processed/gis/regional_chart_plus_6ft_summary.csv
data/processed/gis/regional_decade_metric_summary.csv
data/processed/gis/regional_flood_comparison.csv
data/processed/gis/regional_metric_summary.csv
data/processed/gis/regional_plus_3ft_metric_summary.csv
data/processed/gis/regional_plus_6ft_metric_summary.csv
data/processed/gis/regional_scenario_lookup.csv
data/processed/gis/regional_top_impacted_roads.csv


## Final Freeze Checklist

Use this cell as the last report-readiness check before code freeze.


In [42]:
freeze_checks = pd.DataFrame([
    {
        "check": "Frozen city scope present",
        "expected": len(CITY_EXPORTS),
        "actual": regional["study_area_id"].nunique(),
        "passed": regional["study_area_id"].nunique() == len(CITY_EXPORTS),
    },
    {
        "check": "Full SLR scenario set present",
        "expected": len(FULL_SLR_VALUES),
        "actual": regional["sea_level_rise_ft"].nunique(),
        "passed": regional["sea_level_rise_ft"].nunique() == len(FULL_SLR_VALUES),
    },
    {
        "check": "Regional comparison row count",
        "expected": len(CITY_EXPORTS) * len(FULL_SLR_VALUES),
        "actual": len(regional),
        "passed": len(regional) == len(CITY_EXPORTS) * len(FULL_SLR_VALUES),
    },
    {
        "check": "2100 planning case available",
        "expected": 7,
        "actual": len(year_2100),
        "passed": len(year_2100) == len(CITY_EXPORTS),
    },
    {
        "check": "+6 ft stress view available",
        "expected": 7,
        "actual": len(plus_6ft),
        "passed": len(plus_6ft) == len(CITY_EXPORTS),
    },
    {
        "check": "Priority corridors loaded",
        "expected": "rows > 0",
        "actual": len(priority_corridors) if "priority_corridors" in globals() else 0,
        "passed": "priority_corridors" in globals() and len(priority_corridors) > 0,
    },
])
freeze_checks


,check,expected,actual,passed
0,Frozen city scope present,7,7,True
1,Full SLR scenario set present,12,12,True
2,Regional comparison row count,84,84,True
3,2100 planning case available,7,7,True
4,+6 ft stress view available,7,7,True
5,Priority corridors loaded,rows > 0,575,True


## Key Caveats

- Current report exports include 1-meter GeoPackages for the seven-city scope where available. Earlier coarse regional outputs remain in the GIS folder for comparison, but the refreshed `regional_*` CSVs are the report source.
- Results are screening-level static connected-inundation outputs, not a hydrodynamic simulation.
- Climate-possibility bands are scenario interpretation labels, not annual exceedance probabilities or return periods.
- Damage estimates use a simple depth-based replacement-cost model.
- Non-current SLR damage tables add a total-loss property assumption for flooded buildings, including the `+4`, `+5`, and `+6 ft` flood-on-top-of-SLR stress views.
- Property-value exposure uses city-level ACS median owner-occupied home value as a uniform proxy, not parcel assessment values.
- Regional datum conversion currently uses Sewells Point; production work should evaluate spatially varying tidal datums or VDatum.
- Future decade and `+6 ft` stress-view outputs keep the same event baseline and add sea-level-rise increments; they do not model future storm frequency, rainfall, drainage capacity, shoreline adaptation, or development change.
